# L02 · How a Language Model Produces Probabilities

## Goal

- explain the causal next-token objective
- track token and sequence log-probability shapes
- distinguish prompt, response, and padding masks

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L02:toy:42").hexdigest()
print(f"lesson=L02 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L02 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:709e8a5c9d82a72fe43146d65d9657eea17509dfd65e9e3d5d1f363ee173a420 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: probability/gradients → **causal LM and token masks** → LLM policy

$$\log\pi_\theta(y\mid x)=\sum_t m_t\log p_\theta(y_t\mid x,y_{<t})$$

Causal-LM logits have shape `[batch, time, vocabulary]`, with targets aligned to logits shifted by one position. The prompt is context, not an action, so policy loss keeps only response and EOS positions. Padding carries no reward.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** For response `2`, is the action-mask sum 1, or 2 including EOS? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>This repository includes generation termination as an action, so the answer is 2. Rollout and update must share that contract.</details>

In [2]:
from rl_study.models import TinyCausalLM, TinyTokenizer, build_sequence_batch
from rl_study.models.sequence import response_sequence_log_probs
tokenizer = TinyTokenizer()
model = TinyCausalLM()
sequence_batch = build_sequence_batch(
    ["Q:1+1="], ["2"], tokenizer=tokenizer, max_length=64
)
sequence_logp = response_sequence_log_probs(model, sequence_batch)
print({"input_shape": list(sequence_batch.input_ids.shape),
       "target_shape": list(sequence_batch.action_mask.shape),
       "action_tokens": int(sequence_batch.action_mask.sum()),
       "sequence_logp": round(float(sequence_logp[0].detach()), 3)})

{'input_shape': [1, 9], 'target_shape': [1, 8], 'action_tokens': 2, 'sequence_logp': -149.19}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Mean token log-probability and summed sequence log-probability encode length differently. Explicit masks and reductions in the package API prevent DPO, PPO, and GRPO from silently using different rules.

**Common trap:** Including prompt targets lets long prompts dominate updates. A truth-table test requiring an empty prompt/action-mask intersection catches it. Regression tests: `test_prompt_and_action_mask_truth_table`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert not bool((sequence_batch.prompt_target_mask & sequence_batch.action_mask).any())
assert int(sequence_batch.action_mask.sum()) == 2
print("checks=passed")

checks=passed


**Recall:** What behavior loses its learning signal if EOS is removed from the mask? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** The output shows 8 prediction positions for input length 9 and only 2 action positions. Sequence log-probability sums only those two positions.
- Executable checks: `test_prompt_and_action_mask_truth_table`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L03 temporarily sets sequences aside to study exploration and sampled rewards in their smallest setting.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

## Sources

- `instructgpt-2022` — `docs/sources.yml`